<a href="https://colab.research.google.com/github/rekhaannapurna/Paddy-Disease-Detection/blob/main/Colab/XGBoost_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from pathlib import Path

features_path = Path(
    "/content/drive/MyDrive/Paddy_Disease_Project/features/_MobileNetV2"
)

X_train = np.load(features_path / "X_train.npy")
y_train = np.load(features_path / "y_train.npy")
X_valid = np.load(features_path / "X_valid.npy")
y_valid = np.load(features_path / "y_valid.npy")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (8326, 1280)
y_train: (8326,)
X_valid: (2081, 1280)
y_valid: (2081,)


In [3]:
classes = [
    'bacterial_leaf_blight',
    'bacterial_leaf_streak',
    'bacterial_panicle_blight',
    'blast',
    'brown_spot',
    'dead_heart',
    'downy_mildew',
    'hispa',
    'normal',
    'tungro'
]

In [4]:
!pip install -q xgboost

In [5]:


from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=10,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

# Training
start_time = time.time()
xgb_model.fit(X_train, y_train)
xgb_train_time = time.time() - start_time

# Prediction
start_time = time.time()
xgb_preds = xgb_model.predict(X_valid)
xgb_inference_time = time.time() - start_time

# Evaluation
xgb_accuracy = accuracy_score(y_valid, xgb_preds)

print(f"XGBoost Training time: {xgb_train_time:.4f} seconds")
print(f"XGBoost Inference time: {xgb_inference_time:.4f} seconds")
print(f"\nXGBoost Accuracy: {xgb_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_valid,
    xgb_preds,
    target_names=classes,
    digits=4
))

XGBoost Training time: 583.6434 seconds
XGBoost Inference time: 0.0778 seconds

XGBoost Accuracy: 97.07%

Classification Report:
                          precision    recall  f1-score   support

   bacterial_leaf_blight     0.9691    0.9307    0.9495       101
   bacterial_leaf_streak     0.9844    0.9545    0.9692        66
bacterial_panicle_blight     0.9828    1.0000    0.9913        57
                   blast     0.9694    0.9667    0.9680       360
              brown_spot     0.9624    0.9624    0.9624       213
              dead_heart     0.9931    0.9897    0.9914       292
            downy_mildew     0.9298    0.9381    0.9339       113
                   hispa     0.9727    0.9698    0.9713       331
                  normal     0.9671    0.9848    0.9758       328
                  tungro     0.9683    0.9727    0.9705       220

                accuracy                         0.9707      2081
               macro avg     0.9699    0.9669    0.9683      2081
           

In [6]:
import joblib

model_path = "/content/drive/MyDrive/Paddy_Disease_Project/models"

joblib.dump(
    xgb_model,
    f"{model_path}/model_5_xgboost.pkl"
)

print("XGBoost model saved successfully.")

XGBoost model saved successfully.
